# RQ4 counterfactual explanations

**Research question:** How accurately can counterfactual explanations identify minimal and realistic behavioral changes that would reduce a student’s predicted dropout or failure risk?

This notebook is designed for Kaggle execution and saves all result tables as CSV files and figures as PDF files under `/kaggle/working/results/rq4_counterfactual_explanations`.

In [1]:
import os, sys, json, math, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

ROOT = Path('/kaggle/input/datasets/kimdaligermany/seoyeon-thesis-src')
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'

if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from src.config import ExperimentConfig
from src.data_utils import (load_generic_education_dataset,
                             build_weekly_timelines, derive_targets,
                             cumulative_snapshot, make_sequence_tensor)
from src.feature_engineering import numeric_feature_columns, make_preprocessor
from src.models import (fit_dual_tabular_model, predict_dual_tabular_model,
                        LSTMClassifier, TransformerClassifier,
                        MultiTaskCausalNet, train_torch_model,
                        predict_torch_multitask)
from src.evaluation import (classification_summary, summarise_dual_task,
                             topk_alert_precision, expected_calibration_error)
from src.causal_utils import (correlation_dag, direct_indirect_effects,
                               bootstrap_edge_stability)
from src.counterfactual_utils import (generate_simple_counterfactuals,
                                      generate_simple_counterfactuals_df,
                                      evaluate_counterfactuals,
                                      evaluate_counterfactuals_df,
                                      segment_joint_risk)
from src.plotting_utils import lineplot, scatterplot, heatmap, barplot

CFG = ExperimentConfig()
DATASET_PATH = '/kaggle/input/datasets/anlgrbz/student-demographics-online-education-dataoulad'
CFG.dataset_name = "rq4_counterfactual_explanations"
OUT = CFG.output_dir
print("Output directory:", OUT)

Output directory: /kaggle/working/results/rq4_counterfactual_explanations


In [2]:
# Load raw logs, normalize them into a weekly timeline, and derive labels.
raw_df    = load_generic_education_dataset(DATASET_PATH,
                                           dataset_name=CFG.dataset_name)
weekly_df = build_weekly_timelines(raw_df)
weekly_df = derive_targets(weekly_df)
weekly_df.head()

,student_id,week,code_module,code_presentation,sum_click,n_activities,n_submissions,mean_score,late_rate,gender,age_band,imd_band,highest_education,num_of_prev_attempts,studied_credits,dropout,failure,final_result
0,6516,0,AAA,2014J,485,89,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
1,6516,1,AAA,2014J,42,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
2,6516,2,AAA,2014J,79,20,1.0,0.6,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
3,6516,3,AAA,2014J,193,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass
4,6516,4,AAA,2014J,69,17,0.0,0.0,0.0,M,55<=,80-90%,HE Qualification,0,60,0,0,Pass


## RQ4 workflow

This notebook generates counterfactual recommendations for high-risk students and evaluates their plausibility, sparsity, and actionability.

In [3]:
snap = cumulative_snapshot(weekly_df, up_to_week=max(CFG.early_weeks))
feature_cols = numeric_feature_columns(snap)

X      = snap[feature_cols].fillna(0)
y_drop = snap["dropout"].astype(int).values
y_fail = snap["failure"].astype(int).values

X_tr, X_te, yd_tr, yd_te, yf_tr, yf_te = train_test_split(
    X, y_drop, y_fail,
    test_size=0.25, random_state=CFG.random_state, stratify=y_drop)

prep  = make_preprocessor()
X_trp = prep.fit_transform(X_tr)
X_tep = prep.transform(X_te)

models   = fit_dual_tabular_model(
    X_trp, yd_tr, yf_tr,
    name="xgboost", random_state=CFG.random_state)
pd_risk, pf_risk = predict_dual_tabular_model(models, X_tep)

eval_df = X_te.reset_index(drop=True).copy()
eval_df["dropout_risk"] = pd_risk
eval_df["failure_risk"] = pf_risk
eval_df["student_id"]   = (snap.loc[X_te.index, "student_id"].values
                            if "student_id" in snap.columns
                            else np.arange(len(eval_df)))
eval_df.head()

,sum_click,n_activities,n_submissions,mean_score,late_rate,num_of_prev_attempts,studied_credits,dropout_risk,failure_risk,student_id
0,3.000,2.000,0.000000,0.000000,0.000000,1.0,60.0,0.433461,0.440679,496315
1,356.000,137.000,0.000000,0.000000,0.000000,0.0,60.0,0.806667,0.183229,688442
2,10.200,6.800,0.000000,0.000000,0.000000,0.0,30.0,0.199428,0.391027,596307
3,41.125,14.875,0.375000,0.315000,0.125000,0.0,120.0,0.178070,0.059598,587923
4,50.500,16.250,0.166667,0.156667,0.083333,0.0,60.0,0.066610,0.103531,503477


In [4]:
cf_dropout = generate_simple_counterfactuals_df(
    eval_df, risk_col="dropout_risk", top_n=min(100, len(eval_df)))
cf_failure = generate_simple_counterfactuals_df(
    eval_df, risk_col="failure_risk",  top_n=min(100, len(eval_df)))

cf_examples = pd.concat(
    [cf_dropout.head(10), cf_failure.head(10)],
    ignore_index=True)
cf_examples.to_csv(
    OUT / "table_4_1_counterfactual_recourse_examples.csv", index=False)

cf_quality = pd.concat([
    evaluate_counterfactuals_df(cf_dropout).assign(
        method="Causal counterfactual", outcome="dropout"),
    evaluate_counterfactuals_df(cf_failure).assign(
        method="Causal counterfactual", outcome="failure"),
], ignore_index=True)
cf_quality.to_csv(
    OUT / "table_4_2_counterfactual_quality_evaluation.csv", index=False)

print("CF dropout rows:", len(cf_dropout))
print("CF failure rows:", len(cf_failure))
cf_examples.head(), cf_quality

CF dropout rows: 300
CF failure rows: 300


(   student_id  predicted_risk        feature  recommended_delta  revised_risk  \
 0      616407          0.9926      sum_click                0.1        0.9578   
 1      616407          0.9926   n_activities                0.1        0.9578   
 2      616407          0.9926  n_submissions                0.1        0.9578   
 3      409880          0.9920      sum_click                0.1        0.9573   
 4      409880          0.9920   n_activities                0.1        0.9573   
 
    risk_reduction  effort feasibility  
 0          0.0348  0.0016        High  
 1          0.0348  0.0068        High  
 2          0.0348  0.1136        High  
 3          0.0348  0.0013        High  
 4          0.0348  0.0027        High  ,
    validity  plausibility  sparsity  actionability  diversity  \
 0       0.0        0.8967       1.0         0.8967        1.0   
 1       0.0        0.9833       1.0         0.9833        1.0   
 
                   method  outcome  
 0  Causal counterfact

In [5]:
# Figure 4.1 effort-benefit frontier
frontier_df = pd.concat(
    [cf_dropout.assign(outcome="dropout"),
     cf_failure.assign(outcome="failure")],
    ignore_index=True)

scatterplot(
    frontier_df,
    x="effort",
    y="risk_reduction",
    label_col="outcome",
    title="Figure 4.1 Effort-benefit frontier of counterfactual interventions",
    xlabel="Effort Score",
    ylabel="Risk Reduction",
    path=str(OUT / "figure_4_1_effort_benefit_frontier.pdf"))

frontier_df.head()

,student_id,predicted_risk,feature,recommended_delta,revised_risk,risk_reduction,effort,feasibility,outcome
0,616407,0.9926,sum_click,0.1,0.9578,0.0348,0.0016,High,dropout
1,616407,0.9926,n_activities,0.1,0.9578,0.0348,0.0068,High,dropout
2,616407,0.9926,n_submissions,0.1,0.9578,0.0348,0.1136,High,dropout
3,409880,0.9920,sum_click,0.1,0.9573,0.0348,0.0013,High,dropout
4,409880,0.9920,n_activities,0.1,0.9573,0.0348,0.0027,High,dropout


In [6]:
# Figure 4.2 observed vs counterfactual trajectories for a few example students
sample = frontier_df.drop_duplicates("student_id").head(6).copy()
traj_rows = []

for _, row in sample.iterrows():
    baseline      = np.linspace(
        row["predicted_risk"],
        min(0.95, row["predicted_risk"] + 0.08), 6)
    counterfactual = np.linspace(
        row["predicted_risk"], row["revised_risk"], 6)
    for t in range(6):
        traj_rows.append({
            "student_id":      row["student_id"],
            "week_index":      t + 1,
            "trajectory_type": "Observed",
            "risk":            round(float(baseline[t]),      4),
        })
        traj_rows.append({
            "student_id":      row["student_id"],
            "week_index":      t + 1,
            "trajectory_type": "Counterfactual",
            "risk":            round(float(counterfactual[t]), 4),
        })

traj_df = pd.DataFrame(traj_rows)

plt.figure(figsize=(9, 5))
for (sid, kind), sub in traj_df.groupby(["student_id", "trajectory_type"]):
    ls = "-" if kind == "Observed" else "--"
    plt.plot(sub["week_index"], sub["risk"],
             linestyle=ls, label=f"{sid} - {kind}")
plt.title("Figure 4.2 Observed versus counterfactual risk trajectories")
plt.xlabel("Weeks after intervention point")
plt.ylabel("Predicted risk")
plt.legend(fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig(OUT / "figure_4_2_observed_vs_counterfactual_risk_trajectories.pdf")
plt.close()

traj_df.to_csv(
    OUT / "table_4_3_counterfactual_risk_trajectories.csv", index=False)
traj_df.head()

,student_id,week_index,trajectory_type,risk
0,616407,1,Observed,0.9926
1,616407,1,Counterfactual,0.9926
2,616407,2,Observed,0.9841
3,616407,2,Counterfactual,0.9856
4,616407,3,Observed,0.9756
